In [39]:
import os
os.getcwd()

'/Users/charithabattini/Desktop/Tool/syllabus-policy-stance/notebooks'

In [40]:
os.listdir()

['.DS_Store', 'sample1.ipynb', 'A_Data.ipynb', '.ipynb_checkpoints']

In [41]:
import pandas as pd

In [42]:
df = pd.read_csv("../data/interim/policy_annotation_sample_1000.csv", encoding="latin1")

In [43]:
df.columns

Index(['sentence', 'policy_relevant', 'stance', 'notes', 'Unnamed: 4',
       'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9',
       ...
       'Unnamed: 161', 'Unnamed: 162', 'Unnamed: 163', 'Unnamed: 164',
       'Unnamed: 165', 'Unnamed: 166', 'Unnamed: 167', 'Unnamed: 168',
       'Unnamed: 169', 'Unnamed: 170'],
      dtype='object', length=171)

In [44]:
# Drop all columns that start with "Unnamed"
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

In [45]:
df.columns

Index(['sentence', 'policy_relevant', 'stance', 'notes'], dtype='object')

In [46]:
df["policy_relevant"].value_counts()

policy_relevant
0.0    667
1.0    355
Name: count, dtype: int64

In [47]:
policy_df = df[df["policy_relevant"] == 1].copy()
policy_df = policy_df.reset_index(drop=True)

print(len(policy_df))

355


In [48]:
policy_df["stance"].value_counts()

stance
dis             293
enc              28
conditional      27
conditional       7
Name: count, dtype: int64

In [49]:
policy_df["stance"] = policy_df["stance"].str.lower().str.strip()

policy_df["stance"] = policy_df["stance"].replace({
    "dis": "discouraging",
    "enc": "encouraging"
})

In [50]:
policy_df["stance"].value_counts()

stance
discouraging    293
conditional      34
encouraging      28
Name: count, dtype: int64

In [51]:
policy_df["stance"] = policy_df["stance"].str.lower().str.strip()

policy_df["stance"] = policy_df["stance"].replace({
    "dis": "discouraging",
    "enc": "encouraging"
})

In [52]:
policy_df["stance"].value_counts()

stance
discouraging    293
conditional      34
encouraging      28
Name: count, dtype: int64

In [53]:
policy_df["sentence"].isna().sum()

policy_df = policy_df[policy_df["sentence"].notna()]

policy_df["sentence"] = policy_df["sentence"].astype(str)

In [54]:
policy_df = policy_df.reset_index(drop=True)

In [33]:
X = policy_df["sentence"]
y = policy_df["stance_label"]

In [55]:
policy_df.head()

,sentence,policy_relevant,stance,notes
0,IMPORTANT: Note that Wikipedia is reliable for...,1.0,conditional,NaN
1,Use of sites such as Wikipedia are not allowed...,1.0,discouraging,NaN
2,"Thus, do not use Wikipedia as your primary sou...",1.0,conditional,NaN
3,Examples of plagiarism are (1) submitting an a...,1.0,discouraging,NaN
4,A key critic: [PERSON] [PERSON][PERSON]Reading...,1.0,encouraging,use but not a useful citation


In [56]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [57]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=5000,
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [59]:
print(X_train_tfidf)

  (0, 875)	0.25006235254169035
  (0, 1217)	0.11585182012184182
  (0, 4273)	0.21883375588336076
  (0, 4585)	0.046015371142460675
  (0, 1117)	0.1958193545920781
  (0, 1213)	0.2270479512504077
  (0, 478)	0.12421899811986235
  (0, 2184)	0.19152531250979235
  (0, 897)	0.23710131387862957
  (0, 212)	0.23710131387862957
  (0, 4468)	0.26832991053695915
  (0, 709)	0.25006235254169035
  (0, 3802)	0.25006235254169035
  (0, 2739)	0.26832991053695915
  (0, 4276)	0.26832991053695915
  (0, 1214)	0.25006235254169035
  (0, 2187)	0.26832991053695915
  (0, 4469)	0.26832991053695915
  (0, 3803)	0.26832991053695915
  (1, 4585)	0.06350007612551348
  (1, 990)	0.3019851804390413
  (1, 207)	0.2303202147423351
  (1, 3711)	0.2493065550483746
  (1, 3494)	0.11837905619667737
  (1, 3202)	0.18561056143861948
  :	:
  (280, 2716)	0.28350216728012506
  (281, 4585)	0.06393016871828765
  (281, 265)	0.19187788220660887
  (281, 2856)	0.29438171138440905
  (281, 646)	0.25563381891855946
  (281, 3472)	0.13664596700145257
  (

In [60]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    multi_class="multinomial"
)

model.fit(X_train_tfidf, y_train)

/Users/charithabattini/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


LogisticRegression(class_weight='balanced', max_iter=1000,
                   multi_class='multinomial')

In [61]:
preds = model.predict(X_test_tfidf)

In [62]:
from sklearn.metrics import classification_report

print(classification_report(y_test, preds, target_names=le.classes_))

              precision    recall  f1-score   support

 conditional       0.00      0.00      0.00         7
discouraging       0.83      0.95      0.89        58
 encouraging       0.50      0.17      0.25         6

    accuracy                           0.79        71
   macro avg       0.44      0.37      0.38        71
weighted avg       0.72      0.79      0.75        71

